# doc-structure-mcp 模型选型测试 v2

目标：找到能正确识别红头竖排文字的 OCR 模型组合。

In [ ]:
# 1. 安装
!pip uninstall paddlepaddle -y 2>/dev/null
!pip install paddlepaddle==3.2.2 -q
!pip install paddleocr==3.6.0 -q
import paddle
print(f'Paddle: {paddle.__version__}')

In [ ]:
# 2. 上传测试文件
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f'文件: {image_path}')

In [ ]:
# 3. 测试函数
import cv2, numpy as np, time
from paddleocr import PaddleOCR

results = []

def run_test(name, det, rec, thresh=0.3):
    print(f'\n=== {name}: {det} + {rec} ===')
    try:
        ocr = PaddleOCR(text_detection_model_name=det, text_recognition_model_name=rec,
                       use_textline_orientation=True, lang='ch',
                       text_det_thresh=thresh, text_det_box_thresh=0.5)
        img = cv2.imread(image_path)
        t0 = time.time()
        res = list(ocr.predict(img))
        elapsed = time.time() - t0
        texts = []
        for r in res:
            if r is None: continue
            tl = r.get('rec_texts',[]) if isinstance(r,dict) else getattr(r,'rec_texts',[])
            sl = r.get('rec_scores',[]) if isinstance(r,dict) else getattr(r,'rec_scores',[])
            for i in range(len(tl)):
                t = tl[i].strip()
                s = float(sl[i]) if i < len(sl) else 0.0
                if t and s > 0.3:
                    texts.append(f'{t}({s:.2f})')
        print(f'  {elapsed:.1f}s, {len(texts)}行')
        for t in texts[:20]:
            print(f'  {t}')
        results.append({'name':name,'time':f'{elapsed:.1f}s','lines':len(texts),'texts':texts})
    except Exception as e:
        print(f'  ERROR: {e}')
        results.append({'name':name,'status':str(e)[:60]})

In [ ]:
# 4. 测试不同检测模型（用默认mobile_rec识别）
IMAGE = image_path

run_test('det-v5mobile', 'PP-OCRv5_mobile_det', 'PP-OCRv5_mobile_rec')
run_test('det-v4server', 'PP-OCRv4_server_det', 'PP-OCRv5_mobile_rec')
run_test('det-v3server', 'PP-OCRv3_server_det', 'PP-OCRv5_mobile_rec')
run_test('det-v5mobile+low', 'PP-OCRv5_mobile_det', 'PP-OCRv5_mobile_rec', 0.01)

In [ ]:
# 5. 汇总
print('\n========== 汇总 ==========')
for r in results:
    if 'status' in r:
        print(f'{r["name"]}: ERROR {r["status"]}')
    else:
        print(f'{r["name"]}: {r["time"]} | {r["lines"]}行 | 首行: {r["texts"][0] if r["texts"] else "-"}')